# 🚀 RSDLIR PodTok - GPU Transcription Pipeline

Pipeline này được rút gọn và tập trung hoàn toàn vào việc chạy trên Kaggle để tận dụng **GPU T4 x2** (để bóc băng - Speech-to-Text) cực nhanh.

## 📌 Lưu ý trước khi chạy:
1. Bật **INTERNET ON** trong phần Settings của Kaggle.
2. Chọn Accelerator là **GPU T4 x2** (để Transcribe).
3. Upload thư mục `raw_audio_split` từ máy bạn lên Kaggle dưới dạng Dataset và gắn vào file Notebook này.

In [ ]:
# [BƯỚC 1] Cài đặt thư viện AI bóc băng
!pip install faster-whisper

In [ ]:
import os
import glob
import json
import logging
from pathlib import Path
from faster_whisper import WhisperModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# 🛠️ CẤU HÌNH THƯ MỤC TRÊN KAGGLE
# ==========================================

# Thiết lập thư mục trên Kaggle
KAGGLE_WORKING = Path("/kaggle/working")
# LỖI Ở ĐÂY: Trong ảnh của bạn tên dataset là dấu gạch dưới "_" (podtok_split) thay vì dấu gạch ngang "-"
KAGGLE_INPUT = Path("/kaggle/input/podtok_split") 

# Sửa lại đường dẫn lấy audio
AUDIO_INPUT_DIR = KAGGLE_INPUT / "raw_audio_split"
TRANSCRIPT_DIR = KAGGLE_WORKING / "data" / "transcripts" / "raw_audio_split"

os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

logger.info(f"Thư mục lấy file MP3: {AUDIO_INPUT_DIR}")
logger.info(f"Thư mục lưu file JSON (Transcript): {TRANSCRIPT_DIR}")

## 🎙️ GIAI ĐOẠN 1: TRANSCRIBE NGÔN NGỮ (BÓC BĂNG BẰNG GPU)

In [ ]:
# Khởi tạo mô hình Whisper (Medium) với cấu hình GPU Kaggle cày max tốc độ trên 2 GPU
print("Đang nạp mô hình Whisper vào 2 GPU T4... (Lần đầu sẽ mất vài phút tải)")
# Sử dụng device_index=[0, 1] để yêu cầu CTranslate2 chia tải đều cho cả 2 chiếc T4
model = WhisperModel("medium", device="cuda", device_index=[0, 1], compute_type="float16")
print("Tải xong mô hình AI lên cả 2 kênh GPU!")


In [ ]:
import concurrent.futures
from tqdm.notebook import tqdm

def transcribe_file(audio_path):
    file_name = os.path.basename(audio_path)
    base_name = os.path.splitext(file_name)[0]
    output_path = TRANSCRIPT_DIR / f"{base_name}.json"
    
    if os.path.exists(output_path):
        # Không in ra để tránh trôi progress bar
        return
        
    # ĐỂ CPU NGHỈ NGƠI: Dùng vad_filter để model bỏ qua những khoảng lặng, kết hợp hạ beam_size
    segments, info = model.transcribe(
        str(audio_path), 
        beam_size=2,          # Tối ưu hóa trên GPU: giảm từ 5 xuống 2, suy luận cực nhanh, kết quả 99% giống 5
        language="vi", 
        condition_on_previous_text=False,
        vad_filter=True,      # Ép VAD filter để xử lý silence => giảm tải CPU decoding
        vad_parameters=dict(min_silence_duration_ms=500)
    )
    
    segments_data = []
    full_text_list = []
    
    for segment in segments:
        start_time = round(segment.start, 2)
        end_time = round(segment.end, 2)
        text = segment.text.strip()
        
        segments_data.append({"start": start_time, "end": end_time, "text": text})
        full_text_list.append(text)
        
    final_output = {
        "full_text": " ".join(full_text_list),
        "segments": segments_data
    }
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)
        
    return file_name

# Chạy lặp qua tất cả các file
audio_files = []
for ext in ('*.mp3', '*.m4a', '*.wav'):
    audio_files.extend(glob.glob(os.path.join(AUDIO_INPUT_DIR, '**', ext), recursive=True))

if not audio_files:
    print("❌ KHÔNG TÌM THẤY AUDIO GỐC MANG TỪ MÁY LÊN KAGGLE!")
    print(f"Hãy check lại thư mục Dataset đã mount vào {AUDIO_INPUT_DIR} chưa nhé!")
else:
    print(f"🔎 Tìm thấy {len(audio_files)} file cần bóc băng.")
    
    print("🚀 Đang khởi động tiến trình Song Song kèm Thanh tiến độ...")
    # Dùng tqdm kết hợp ThreadPool để hiển thị thanh tiến trình
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = [executor.submit(transcribe_file, audio) for audio in audio_files]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(audio_files), desc="Đang bóc băng", unit="file"):
            try:
                result = future.result()
            except Exception as e:
                print(f"Lỗi khi xử lý: {e}")
        
print("\n🎯 HOÀN TẤT TRANSCRIBE TOÀN BỘ!")


In [ ]:
# BƯỚC CUỐI: Đóng gói Zip kết quả
!zip -r podtok_transcripts.zip /kaggle/working/data/transcripts
print("✅ Đã nén xong! Hãy vào thư mục Output của Kaggle để tải file podtok_transcripts.zip về máy!")